# Análise Textual de Contratos PNCP

Sequência didática aplicada aos contratos do PNCP:

1. Pré-processamento de textos + Bag-of-Words/TF-IDF (1,2,3-gramas)
2. Rede k-NN sobre TF-IDF + agrupamento por Label Propagation
3. Descritores por cluster (termos de maior TF-IDF)
4. Embeddings contextuais (Sentence-BERT multilíngue)
5. Rede k-NN sobre embeddings + clusters
6. Geração de indicadores por cluster via LLM (Google Gemini)
7. Classificação semissupervisionada por regularização em grafo
8. Agente LLM com ferramenta de busca semântica

Pré-requisito: ter o arquivo `contratos.parquet` no Google Drive
(produzido por uma coleta anterior).

## Bibliotecas

In [ ]:
!pip install -q sentence-transformers networkx plotly nltk
!pip install -q google-generativeai
!pip install -q langchain langchain-community langchain-google-genai langchain-core
!pip install -q pyarrow pandas

In [ ]:
import pandas as pd
import numpy as np

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics.pairwise import cosine_similarity

import networkx as nx
from networkx.algorithms import community

import plotly.graph_objects as go

## Métodos de apoio

- `meu_tokenizador`: caixa-baixa + tokenização + remove stopwords (PT-BR) + radicalização RSLP.
- `get_cluster_descriptors`: para um cluster, devolve os termos com maior soma TF-IDF.
- `show_graph_cluster`/`show_graph_regularization`: visualização interativa Plotly.
- `simple_regularization`: classificação semissupervisionada por propagação iterativa sobre o grafo.

In [ ]:
STOP_PT = set(nltk.corpus.stopwords.words('portuguese'))
STEMMER = RSLPStemmer()

def remove_stopwords(texto, stop_words=STOP_PT):
    s = str(texto).lower()
    tokens = word_tokenize(s, language='portuguese')
    return [w for w in tokens
            if w not in stop_words and w.isalnum() and not w.isdigit() and len(w) > 2]

def stemming(tokens, stemmer=STEMMER):
    return [stemmer.stem(w) for w in tokens]

def meu_tokenizador(doc):
    return stemming(remove_stopwords(doc))

def get_cluster_descriptors(VSM, df_documentos, cluster_id, max_terms=5):
    sub = df_documentos[df_documentos.cluster == cluster_id]['text']
    df_desc = pd.DataFrame({
        'word': VSM.get_feature_names_out(),
        'tfidf_sum': VSM.transform(sub).toarray().sum(axis=0),
    }).sort_values('tfidf_sum', ascending=False)
    return len(sub), df_desc[df_desc.tfidf_sum > 0].head(max_terms).word.tolist()

In [ ]:
def _arestas_xy(G):
    ex, ey = [], []
    for u, v in G.edges():
        x0, y0 = G.nodes[u]['pos']
        x1, y1 = G.nodes[v]['pos']
        ex += [x0, x1, None]
        ey += [y0, y1, None]
    return ex, ey

def show_graph_cluster(G):
    ex, ey = _arestas_xy(G)
    edge_trace = go.Scatter(x=ex, y=ey,
                            line=dict(width=2, color='#888'),
                            hoverinfo='none', mode='lines')
    nx_, ny_, txt, cor = [], [], [], []
    for n in G.nodes():
        x, y = G.nodes[n]['pos']
        nx_.append(x); ny_.append(y)
        txt.append(G.nodes[n].get('text', str(n)))
        cor.append(G.nodes[n].get('cluster', 0))
    node_trace = go.Scatter(x=nx_, y=ny_, mode='markers', hoverinfo='text',
                            text=txt,
                            marker=dict(size=10, line_width=2, color=cor))
    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(showlegend=False, hovermode='closest',
                                     margin=dict(b=20, l=5, r=5, t=40),
                                     xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                                     yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
    fig.show()

def show_graph_regularization(G):
    ex, ey = _arestas_xy(G)
    edge_trace = go.Scatter(x=ex, y=ey, line=dict(width=1, color='#888'),
                            hoverinfo='none', mode='lines')
    nx_, ny_, txt, cor = [], [], [], []
    for n in G.nodes():
        x, y = G.nodes[n]['pos']
        nx_.append(x); ny_.append(y)
        f = G.nodes[n].get('f', np.array([0.0]))
        txt.append(f'{f}<br>{G.nodes[n].get("text", str(n))}')
        cor.append(float(f[0]))
    node_trace = go.Scatter(x=nx_, y=ny_, mode='markers', hoverinfo='text',
                            text=txt,
                            marker=dict(showscale=True, colorscale='Reds',
                                        size=10, line_width=2, color=cor))
    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(showlegend=False, hovermode='closest',
                                     margin=dict(b=20, l=5, r=5, t=40),
                                     xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                                     yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
    fig.show()

In [ ]:
def simple_regularization(G, labels, max_iter=30):
    """Propagação iterativa: cada nó recebe a média dos vizinhos; nós
    rotulados (seeds) ficam fixos. Resultado em G.nodes[n]['f']."""
    for n in G.nodes():
        G.nodes[n]['f'] = np.array([0.0])
        if n in labels:
            G.nodes[n]['y'] = np.array([1.0])
            G.nodes[n]['f'] = np.array([1.0])
    for it in range(max_iter):
        diff = 0.0
        for node in G.nodes():
            if node in labels:
                continue
            viz = list(G.neighbors(node))
            if not viz:
                continue
            f_new = np.mean([G.nodes[v]['f'] for v in viz], axis=0)
            diff += float(np.linalg.norm(G.nodes[node]['f'] - f_new))
            G.nodes[node]['f'] = f_new
            if 'y' in G.nodes[node]:
                G.nodes[node]['f'] = G.nodes[node]['y']
        print(f'Iteração #{it+1} Q(F)={diff:.4f}')

## Modelo LLM (Google Gemini)

Configura a API do Gemini lendo a chave do `userdata` do Colab.

**Como configurar a chave:**
1. No Colab, clique no ícone de chave 🔑 (Secrets) na barra lateral esquerda
2. Adicione um secret chamado `GOOGLE_API_KEY` com sua chave da AI Studio
3. Marque "Notebook access" para que ele apareça via `userdata.get(...)`

Pegue uma chave gratuita em https://aistudio.google.com/app/apikey

In [ ]:
import google.generativeai as genai
from google.colab import userdata

def llm_local():
    GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print('Gemini API configured.')

def llm_task(model, system, prompt):
    gemini_model = genai.GenerativeModel(model_name=model)
    response = gemini_model.generate_content(
        [system, prompt]
    )
    return response.text

llm_local()


## Base textual — contratos PNCP

Lê o consolidado `contratos.parquet` diretamente do Drive. **Não precisa**
do pacote `pncp` — basta o arquivo coletado previamente.

Para a parte exploratória usamos uma amostra estratificada de até 3000 contratos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Ajuste o caminho conforme onde você guardou os dados no Drive
CAMINHO_PARQUET = '/content/drive/MyDrive/PNCP_TCC/dados/coleta/contratos.parquet'

df_contratos = pd.read_parquet(
    CAMINHO_PARQUET,
    columns=['numeroControlePNCP', 'objeto', 'rotulo',
             'razaoSocialOrgao', 'municipioNome', 'valor'],
)
df_contratos = df_contratos.rename(columns={'objeto': 'text'})
df_contratos = df_contratos.dropna(subset=['text'])
df_contratos = df_contratos[df_contratos['text'].str.len() > 20].reset_index(drop=True)

# Amostra estratificada por rótulo (até 3000 contratos)
ALVO = 3000
if len(df_contratos) > ALVO:
    df_amostra = (df_contratos.groupby('rotulo', group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), ALVO // 3),
                                              random_state=42)))
    df_amostra = df_amostra.reset_index(drop=True)
else:
    df_amostra = df_contratos.reset_index(drop=True)

print(f'Total na base: {len(df_contratos):,} | amostra desta análise: {len(df_amostra):,}')
df_amostra.head()

Mounted at /content/drive


## Modelo Espaço-Vetorial + TF-IDF

TF-IDF com tokenizador customizado (stopwords PT + stemmer RSLP), corte por
frequência mínima de documentos (`min_df`).

In [ ]:
VSM = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None, min_df=3)
X = VSM.fit_transform(df_amostra['text'])
print(f'X: {X.shape[0]:,} docs x {X.shape[1]:,} termos (nnz={X.nnz:,})')

df_word_tfidfs = pd.DataFrame({
    'word': VSM.get_feature_names_out(),
    'tfidf_sum': X.toarray().sum(axis=0),
}).sort_values('tfidf_sum', ascending=False)
df_word_tfidfs.head(50)

## N-gramas (bigramas e trigramas)

Mostra termos compostos relevantes (ex.: *projeto básico*, *obra civil*, *manutenção predial*).

In [ ]:
VSM_bi = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                          min_df=3, ngram_range=(2, 2))
X_bi = VSM_bi.fit_transform(df_amostra['text'])
df_bi = pd.DataFrame({
    'word': VSM_bi.get_feature_names_out(),
    'tfidf_sum': X_bi.toarray().sum(axis=0),
}).sort_values('tfidf_sum', ascending=False)
df_bi.head(30)

In [ ]:
VSM_tri = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                           min_df=3, ngram_range=(3, 3))
X_tri = VSM_tri.fit_transform(df_amostra['text'])
df_tri = pd.DataFrame({
    'word': VSM_tri.get_feature_names_out(),
    'tfidf_sum': X_tri.toarray().sum(axis=0),
}).sort_values('tfidf_sum', ascending=False)
df_tri.head(30)

## Rede k-NN sobre TF-IDF (1+2-gramas) + Label Propagation

In [ ]:
VSM = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                       min_df=3, ngram_range=(1, 2))
X = VSM.fit_transform(df_amostra['text'])
A = kneighbors_graph(X, n_neighbors=5, metric='cosine')
G = nx.Graph(A)

cluster_id = 0
for clusters in community.label_propagation_communities(G):
    for doc_id in clusters:
        G.nodes[doc_id]['cluster'] = cluster_id
    cluster_id += 1

df_amostra['cluster'] = [G.nodes[i].get('cluster', -1) for i in df_amostra.index]
df_amostra.head()

In [ ]:
# Estatísticas por cluster
df_stats = (df_amostra[['text', 'cluster']].groupby('cluster').count()
            .sort_values(by='text', ascending=False)
            .rename(columns={'text': 'num_docs'}))
df_stats.head(20)

### Descritores de cluster (termos com maior TF-IDF)

In [ ]:
QTD_TOPICOS = 15
linhas = []
for c in df_stats.head(QTD_TOPICOS).index:
    n_docs, descs = get_cluster_descriptors(VSM, df_amostra, c)
    linhas.append([c, n_docs, descs])
df_descriptors = pd.DataFrame(linhas, columns=['cluster', 'num_docs', 'descriptors'])
df_descriptors

### Subgrafo dos top-N clusters + visualização

In [ ]:
selected = df_descriptors.cluster.tolist()
G2 = G.copy()
for n in list(G2.nodes()):
    if G2.nodes[n].get('cluster') not in selected:
        G2.remove_node(n)

pos = nx.spring_layout(G2, seed=42)
for n in G2.nodes():
    G2.nodes[n]['pos'] = pos[n]

for i, row in df_amostra.iterrows():
    if i in G2.nodes:
        descs = df_descriptors[df_descriptors.cluster == G2.nodes[i]['cluster']]
        cabec = str(descs.descriptors.tolist()[0]) if not descs.empty else ''
        G2.nodes[i]['text'] = (str(row.get('rotulo', '')) + ' | ' + cabec +
                                  '<br>' + str(row['text'])[:120])

show_graph_cluster(G2)

## Embeddings contextuais (Sentence-BERT multilíngue)

DistilBERT multilíngue ajustado para similaridade semântica entre sentenças.

In [ ]:
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('distiluse-base-multilingual-cased-v1')
X_emb = sbert.encode(df_amostra['text'].tolist(), show_progress_bar=True,
                       batch_size=64, convert_to_numpy=True,
                       normalize_embeddings=True)
X_emb.shape

In [ ]:
A_emb = kneighbors_graph(X_emb, n_neighbors=5, metric='cosine')
G_emb = nx.Graph(A_emb)

cid = 0
for clusters in community.label_propagation_communities(G_emb):
    for doc_id in clusters:
        G_emb.nodes[doc_id]['cluster'] = cid
    cid += 1
df_amostra['cluster_sbert'] = [G_emb.nodes[i].get('cluster', -1)
                                  for i in df_amostra.index]

df_stats_emb = (df_amostra[['text', 'cluster_sbert']]
                .groupby('cluster_sbert').count()
                .sort_values('text', ascending=False)
                .rename(columns={'text': 'num_docs'}))
df_stats_emb.head(20)

### Subgrafo dos top-50 clusters semânticos + visualização

In [ ]:
selected_emb = list(df_stats_emb.head(50).index)
G_emb2 = G_emb.copy()
for n in list(G_emb2.nodes()):
    if G_emb2.nodes[n].get('cluster') not in selected_emb:
        G_emb2.remove_node(n)

pos = nx.spring_layout(G_emb2, seed=42)
for n in G_emb2.nodes():
    G_emb2.nodes[n]['pos'] = pos[n]

for i, row in df_amostra.iterrows():
    if i in G_emb2.nodes:
        G_emb2.nodes[i]['text'] = (str(row.get('rotulo', '')) + '<br>' +
                                       str(row['text'])[:120])

show_graph_cluster(G_emb2)

## Geração de indicadores por cluster via LLM

Para cada cluster grande, o modelo lê uma amostra dos objetos e devolve um
indicador JSON estruturado: nome, categoria e resumo.

In [ ]:
SYSTEM_INDICADOR = '''
Você é analista de inteligência analítica especializado em contratações
públicas brasileiras (Lei 14.133/2021).

Recebe uma amostra de objetos de contratos agrupados por similaridade
semântica (cluster). Sua tarefa: identificar UM indicador-chave que
sintetize o padrão presente no cluster, especialmente quando houver indício
de subenquadramento (contratos rotulados como "serviços gerais" que
deveriam ser "engenharia/obras").

Responda APENAS no JSON:
{
  "indicador": {
    "nome": "Nome curto e descritivo",
    "categoria": "obra civil | manutenção | serviço técnico de engenharia | vigilância/limpeza | fornecimento | transporte | outro",
    "resumo": "1-2 frases sobre o padrão observado e por que importa para auditoria/controle",
    "indicio_subenquadramento": "alto | medio | baixo | nenhum",
    "recomendacao": "uma ação concreta"
  }
}
'''

In [ ]:
TOP_N_CLUSTERS = 10
N_OBJETOS_POR_CLUSTER = 8
# Modelos Gemini disponíveis: gemini-1.5-flash (rápido/barato),
# gemini-1.5-pro (qualidade máxima), gemini-2.0-flash-exp (mais recente).
# 'gemini-pro' antigo foi descontinuado — use os modelos acima.
MODELO_LLM = 'gemini-1.5-flash'

indicadores = []
for cid in df_stats_emb.head(TOP_N_CLUSTERS).index:
    sub = df_amostra[df_amostra.cluster_sbert == cid].head(N_OBJETOS_POR_CLUSTER)
    objetos = '\n'.join(f'- {str(o)[:300]}' for o in sub['text'].tolist())
    prompt = (f'Cluster #{cid} ({len(sub)} objetos amostrados).\n\n'
              f'Objetos:\n{objetos}\n\nGere o indicador no JSON.')
    try:
        resp = llm_task(MODELO_LLM, SYSTEM_INDICADOR, prompt)
    except Exception as e:
        print(f'[cluster {cid}] LLM falhou: {e}')
        continue
    print(f'\n========= Cluster #{cid} (n={len(sub)}) =========')
    print(resp)
    indicadores.append({'cluster': cid, 'n_docs': len(sub), 'resposta': resp})

## Classificação semissupervisionada por regularização em grafo

Dado um pequeno conjunto de seeds confirmados como subenquadramento de
engenharia (ex.: contratos rotulados 'geral' contendo termos óbvios como
*ART*, *obra*, *projeto básico*), propagamos a "confiança" pelos vizinhos
do grafo k-NN. O resultado em `G.nodes[n]['f']` é um score 0–1.

In [ ]:
TERMOS_OBVIOS = ('art ', 'rrt ', 'crea', 'projeto básico', 'projeto basico',
                  'obra de engenharia', 'reforma predial', 'pavimentação',
                  'pavimentacao', 'memorial descritivo', 'abnt nbr')

labels = {}
for n in G_emb2.nodes():
    objeto = str(df_amostra.loc[n, 'text']).lower()
    rotulo = str(df_amostra.loc[n, 'rotulo'])
    if rotulo == 'geral' and any(t in objeto for t in TERMOS_OBVIOS):
        labels[n] = 1

print(f'Seeds encontradas (geral + termo óbvio): {len(labels)}')
for n in list(labels)[:5]:
    print(f'  #{n}: {str(df_amostra.loc[n, "text"])[:120]}')

In [ ]:
if labels:
    simple_regularization(G_emb2, labels, max_iter=30)
    show_graph_regularization(G_emb2)
else:
    print('Sem seeds — refine TERMOS_OBVIOS ou aumente a amostra.')

### Filtrando candidatos por limiar

In [ ]:
LIMIAR = 0.5
candidatos = []
for n in G_emb2.nodes():
    f = G_emb2.nodes[n].get('f', np.array([0.0]))
    if float(f[0]) > LIMIAR and n not in labels:
        candidatos.append(n)

df_candidatos = df_amostra.loc[candidatos, ['numeroControlePNCP', 'rotulo',
                                              'text', 'municipioNome', 'valor']]
print(f'Candidatos com score > {LIMIAR}: {len(df_candidatos)}')
df_candidatos.head(20)

## Agente LLM com ferramenta de busca semântica

O agente recebe perguntas em português, escolhe a ferramenta `busca_semantica_grafo`
quando precisa de contratos similares ao texto da consulta, e responde com
base no contexto recuperado.

In [ ]:
import os
from langchain_core.tools import Tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import initialize_agent
from langchain.agents.agent_types import AgentType

def busca_semantica_grafo(query=''):
    if not query:
        return 'Por favor, forneça uma consulta.'
    q_emb = sbert.encode([query], normalize_embeddings=True)[0]
    sims = cosine_similarity([q_emb], X_emb)[0]
    idx = int(np.argmax(sims))
    viz = list(G_emb.neighbors(idx)) + [idx]
    textos = []
    for v in viz[:5]:
        row = df_amostra.loc[v]
        textos.append(f"[{row.get('rotulo', '?')}] {row.get('numeroControlePNCP', '?')}: "
                       f"{str(row['text'])[:300]}")
    return (f'Contratos similares a "{query}":\n\n' +
            '\n\n===\n\n'.join(textos) +
            '\n\n=== Fim ===')

ferramenta_busca = Tool(
    name='busca_semantica_grafo',
    func=busca_semantica_grafo,
    description='Busca contratos PNCP similares a um texto livre. Use quando '
                'a pergunta exige exemplos concretos de contratos.',
)

# ChatGoogleGenerativeAI lê GOOGLE_API_KEY do ambiente
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

agente = initialize_agent(
    tools=[ferramenta_busca],
    llm=ChatGoogleGenerativeAI(model='gemini-1.5-flash', temperature=0),
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
)

In [ ]:
pergunta = ('Encontre contratos rotulados como serviços gerais que mencionem '
            'manutenção elétrica predial e podem exigir ART do CREA. '
            'Quais padrões aparecem?')
resposta = agente.run(pergunta)
print('\n--- RESPOSTA FINAL ---\n')
print(resposta)